Cell Number 1

## RAG Baseline Pipeline

LangChain 기반 RAG 파이프라인 베이스라인 노트북입니다.
기존 gpt-3.5-turbo 에서 gpt-4o-mini 로 모델을 업그레이드하였으며,
이후 02_RAG_LangGraph.ipynb 와의 성능 비교를 위한 기준 버전입니다.

전체 구성:

- Step 0: 환경 설정
- Step 1: LLM 설정 (gpt-4o-mini)
- Step 2: Document Loader (Demian.pdf)
- Step 3: Text Splitter (RecursiveCharacterTextSplitter + tiktoken)
- Step 4: Embedding (text-embedding-3-small)
- Step 5: VectorStore (ChromaDB)
- Step 6: Retriever + QA Chain
- Step 7: 테스트 질의응답
- Step 8: 결과 저장 (RAGAS 평가용)

Cell Number 2

## Step 0: 환경 설정

필요한 패키지를 설치합니다.
이미 설치되어 있다면 아래 셀의 주석을 그대로 두어도 됩니다.

In [1]:
# Cell Number 3
# 필요 패키지 설치 (최초 1회만 실행)
# !pip install langchain langchain-openai langchain-community
# !pip install chromadb pypdf tiktoken
# !pip install python-dotenv numpy pandas

In [2]:
# Cell Number 4
# 환경변수 로드 (.env 파일에서 OPENAI_API_KEY 읽기)
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 등록 여부 확인
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("OPENAI_API_KEY 로드 완료")
else:
    print("OPENAI_API_KEY 가 없습니다. .env 파일을 확인하세요.")

OPENAI_API_KEY 로드 완료


Cell Number 5

## Step 1: LLM 설정

기존 gpt-3.5-turbo 에서 gpt-4o-mini 로 업그레이드합니다.

변경 이유:
- gpt-4o-mini 는 gpt-3.5-turbo 보다 성능이 높고 비용은 비슷하거나 더 저렴합니다.
- 한국어 처리 능력이 더 우수합니다.
- RAGAS 평가에서 Judge LLM 역할에도 적합합니다.
- temperature=0 으로 설정하여 일관성 있는 답변을 유도합니다.

In [3]:
# Cell Number 6
from langchain_openai import ChatOpenAI

# LLM 선언: gpt-4o-mini (기존 gpt-3.5-turbo 에서 업그레이드)
# temperature=0: 창의성 억제, 사실 기반 답변 유도
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 간단한 동작 확인
response = llm.invoke("안녕하세요. 당신의 모델명은 무엇인가요?")
print(response.content)

/Users/macminim4/PyCharmMiscProject/RAG/rag-practice/ragvenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


안녕하세요! 저는 OpenAI의 GPT-3 모델입니다. 어떻게 도와드릴까요?


Cell Number 7

## Step 2: Document Loader

PyPDFLoader 를 사용하여 Demian.pdf 파일을 불러옵니다.
각 페이지가 Document 객체(page_content + metadata) 로 변환됩니다.

Demian.pdf: 헤르만 헤세의 소설 데미안 (영문판, 182페이지)

In [4]:
# Cell Number 8
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 경로 (상위 폴더에 위치)
PDF_PATH = "../Demian.pdf"

# PDF 로드: 각 페이지를 Document 객체로 변환
loader = PyPDFLoader(PDF_PATH)
pages = loader.load_and_split()

print(f"총 {len(pages)} 페이지 로드됨")

총 182 페이지 로드됨


In [5]:
# Cell Number 9
# 로드된 Document 구조 확인 (page_content + metadata)
print("페이지 본문 미리보기:")
print("-" * 40)
print(pages[10].page_content[:300])
print("\n메타데이터:")
print(pages[10].metadata)

페이지 본문 미리보기:
----------------------------------------
TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at

메타데이터:
{'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '../Demian.pdf', 'total_pages': 182, 'page': 10, 'page_label': '11'}


Cell Number 10

## Step 3: Text Splitter

긴 문서를 작은 Chunk 로 분할합니다.
LLM 의 토큰 제한을 극복하고 검색 정확도를 높이기 위한 단계입니다.

사용 방식:
- RecursiveCharacterTextSplitter: 문단, 문장 단위로 재귀적 분할 (권장 방식)
- tiktoken: 글자 수 대신 실제 토큰 수 기준으로 분할하여 정확도 향상

파라미터 설정:
- chunk_size=500: 청크 하나당 최대 500 토큰
- chunk_overlap=50: 청크 간 50 토큰 중복 (문맥 단절 방지)

In [6]:
# Cell Number 11
import tiktoken

# tiktoken 기반 토큰 길이 계산 함수
# cl100k_base: GPT-4, GPT-3.5 에서 사용하는 인코딩 방식
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text: str) -> int:
    """텍스트의 토큰 수를 반환하는 함수"""
    tokens = tokenizer.encode(text)
    return len(tokens)

# 글자 수 vs 토큰 수 차이 확인
sample = "Demian looked at Sinclair with his calm, knowing eyes."
print(f"글자 수: {len(sample)}")
print(f"토큰 수: {tiktoken_len(sample)}")

글자 수: 54
토큰 수: 12


In [7]:
# Cell Number 12
from langchain.text_splitter import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter: 단락 > 문장 > 단어 순서로 재귀 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # 청크 최대 토큰 수
    chunk_overlap=50,      # 청크 간 중복 토큰 수 (문맥 단절 방지); 앞 청크의 마지막 내용과 뒤 청크의 시작 내용이 50토큰(약 10% 정도) 겹치게 함
    length_function=tiktoken_len  # 토큰 수 기준 분할
)

# 전체 페이지를 청크 단위로 분할
docs = text_splitter.split_documents(pages)

print(f"원본 페이지 수: {len(pages)}")
print(f"분할된 청크 수: {len(docs)}")
print(f"\n첫 번째 청크 미리보기:")
print("-" * 40)
print(docs[0].page_content[:200])

원본 페이지 수: 182
분할된 청크 수: 182

첫 번째 청크 미리보기:
----------------------------------------
DEMIAN 
• 
Downloaded from https://www.holybooks.com


Cell Number 13

## Step 4: Embedding

텍스트를 고차원 숫자 벡터로 변환합니다.
의미가 유사한 텍스트는 벡터 공간에서 가까운 위치에 배치됩니다.

- 모델: text-embedding-3-small (OpenAI)
- 차원: 1536 차원 벡터

In [8]:
# Cell Number 14
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델 선언
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 임베딩 동작 확인 (벡터 차원 출력)
test_vector = embedding_model.embed_query("What does Demian look like?")
print(f"임베딩 벡터 차원: {len(test_vector)}")
print(f"벡터 앞 5개 값: {test_vector[:5]}")

임베딩 벡터 차원: 1536
벡터 앞 5개 값: [0.014311072416603565, -0.013790171593427658, -0.019355587661266327, -0.004448221065104008, 0.0159971471875906]


Cell Number 15

## Step 5: VectorStore

임베딩된 청크를 ChromaDB 에 저장합니다.
이후 질문이 들어오면 이 저장소에서 가장 유사한 청크를 검색합니다.

ChromaDB: 로컬에서 바로 사용 가능한 벡터 데이터베이스

In [9]:
# Cell Number 16
from langchain_community.vectorstores import Chroma

# 청크를 임베딩하여 ChromaDB 에 저장 (시간 소요)
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

print(f"{len(docs)} 개 청크가 VectorStore 에 저장되었습니다.")

# 유사도 검색 테스트
test_query = "What does Demian look like?"
results = vectorstore.similarity_search(test_query, k=2)
print(f"\n검색 결과 (LLM 없이 청크만 반환):")
print("-" * 40)
print(results[0].page_content[:200])

182 개 청크가 VectorStore 에 저장되었습니다.

검색 결과 (LLM 없이 청크만 반환):
----------------------------------------
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directe


Cell Number 17

## Step 6: Retriever + QA Chain

Retriever 와 LLM 을 연결하여 최종 RAG 파이프라인을 완성합니다.

파이프라인 흐름:
질문 -> 벡터 변환 -> 유사 청크 검색 -> LLM 으로 전달 -> 답변 생성

파라미터 설명:
- chain_type=stuff: 검색된 문서를 그대로 프롬프트에 삽입하는 방식
- search_type=mmr: 유사도와 다양성을 함께 고려하는 검색 방식
- k=3: LLM 에 전달할 최종 문서 수
- fetch_k=10: 후보로 가져올 문서 수 (k 보다 크게 설정)

In [10]:
# Cell Number 18
from langchain.chains import RetrievalQA

# Retriever 설정: MMR 검색 방식
retriever = vectorstore.as_retriever(
    search_type="mmr", #유사도와 다양성을 함께 고려하는 검색 방식
    search_kwargs={"k": 3, "fetch_k": 10}
)

# RAG QA Chain 구성: Retriever + LLM 연결
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True  # 출처 문서도 함께 반환
)

print("RAG QA Chain 구성 완료")
print(f"  - 모델: gpt-4o-mini")
print(f"  - 검색 방식: MMR") #유사도와 다양성을 함께 고려하는 검색 방식
print(f"  - 반환 문서 수: k=3, fetch_k=10")

RAG QA Chain 구성 완료
  - 모델: gpt-4o-mini
  - 검색 방식: MMR
  - 반환 문서 수: k=3, fetch_k=10


Cell Number 19

## Step 7: 테스트 질의응답

Demian.pdf 내용을 기반으로 5개의 테스트 질문을 실행합니다.
각 답변과 참조 출처를 함께 출력합니다.

이 결과는 이후 RAGAS 평가의 기준 데이터로 저장됩니다.

In [11]:
# Cell Number 20
# 테스트 질문 목록 정의
# 동일한 질문이 02_RAG_LangGraph.ipynb 에서도 사용됩니다.
TEST_QUESTIONS = [
    "How does Demian look like?",
    "What is the relationship between Sinclair and Demian?",
    "Who is Frau Eva and what role does she play?",
    "What does the bird breaking out of the egg symbolize?",
    "How does Sinclair's worldview change throughout the novel?"
]

def run_rag_query(query: str) -> dict:
    """RAG 파이프라인 실행 함수: 질문을 받아 답변과 참조 문서를 반환"""
    result = qa_chain.invoke(query)#qa_chain : Cell 18에서 정의 "검색기(Retriever)"와 "생성기(LLM)"를 하나로 묶은 최종 실행 객체입니다.
    answer = result["result"]
    # 검색된 문서의 본문만 추출하여 리스트로 구성
    contexts = [doc.page_content for doc in result["source_documents"]] #텍스트 본문에 해당하는 .page_content 값만 가져옵니다.
    return {"question": query, "answer": answer, "contexts": contexts}
# run_rag_query 함수의 최종 결과물을 파이썬 딕셔너리(Dictionary) 형태로 구조화하여 반환하는 부분입니다. 이는 RAG 성능 평가 프레임워크인 RAGAS의 데이터 요구 사항을 충족하기 위한 표준 포맷입니다.

print(f"테스트 질문 {len(TEST_QUESTIONS)} 개 준비 완료")

테스트 질문 5 개 준비 완료


In [12]:
# Cell Number 21
# 테스트 질문 실행 및 결과 수집
baseline_results = []

for i, question in enumerate(TEST_QUESTIONS):
    print(f"[{i+1}/{len(TEST_QUESTIONS)}] 질문: {question}")
    result = run_rag_query(question)
    baseline_results.append(result)

    # 답변 출력
    print(f"답변: {result['answer']}")
    print(f"참조 청크 수: {len(result['contexts'])}")
    print("-" * 60)

[1/5] 질문: How does Demian look like?
답변: Demian's appearance is described as elegant and at ease, with a face that seems neither masculine nor childish, but rather timeless and bearing elements of both. The narrator perceives something different about him, almost a feminine quality, and notes that his face could be seen as handsome or attractive, yet also potentially repelling. Overall, he is depicted as unimaginarily different from others, with an aura that suggests he is like an animal, a spirit, or an image.
참조 청크 수: 3
------------------------------------------------------------
[2/5] 질문: What is the relationship between Sinclair and Demian?
답변: The relationship between Sinclair and Demian is complex and deeply intertwined. Demian serves as a mentor and guide for Sinclair, helping him explore his inner self and understand his identity. Sinclair is drawn to Demian's ideas and presence, which challenge him to think differently about life and his own fate. Demian represents a figure of

Cell Number 22

## Step 8: 결과 저장 (RAGAS 평가용)

테스트 결과를 JSON 파일로 저장합니다.
03_RAG_RAGAS_Evaluation.ipynb 에서 이 파일을 불러와 평가에 사용합니다.

저장 형식:
- question: 사용자 질문
- answer: RAG 가 생성한 답변
- contexts: 검색된 문서 청크 리스트

In [13]:
# Cell Number 23
import json

# RAGAS 호환 형식으로 변환
# user_input, response, retrieved_contexts 는 RAGAS 에서 요구하는 키 이름
ragas_format = {
    "user_input": [r["question"] for r in baseline_results],
    "response": [r["answer"] for r in baseline_results],
    "retrieved_contexts": [r["contexts"] for r in baseline_results]
}

# 결과 파일 저장
OUTPUT_PATH = "baseline_results.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(ragas_format, f, ensure_ascii=False, indent=2)

print(f"결과 저장 완료: {OUTPUT_PATH}")
print(f"  - 질문 수: {len(ragas_format['user_input'])}")
print(f"  - 다음 단계: 03_RAG_RAGAS_Evaluation.ipynb 에서 평가 진행")

결과 저장 완료: baseline_results.json
  - 질문 수: 5
  - 다음 단계: 03_RAG_RAGAS_Evaluation.ipynb 에서 평가 진행
